In [0]:
%run "../00_Configuration/00_parametres"

In [0]:
import pandas as pd
import re
from pyspark.sql.functions import current_timestamp, lit

# La fonction "Super Ingestion" pour Excel
def ingest_excel_to_bronze(file_name, table_name):
    try:
        # Pandas lit le fichier depuis le Volume (sans dbfs:)
        chemin_local = f"{PATH_BRUT}{file_name}".replace("dbfs:", "")
        pdf = pd.read_excel(chemin_local)
        
        # --- NETTOYAGE DES COLONNES (Regex) ---
        # Remplace les espaces, virgules, et caractères spéciaux par des underscores
        pdf.columns = [re.sub(r'[ ,;{}()\n\t=°]', '_', str(col)) for col in pdf.columns]
        
        # Conversion en DataFrame Spark (en forçant le type texte)
        df = spark.createDataFrame(pdf.astype(str))
        
        # Audit
        df = df.withColumn("date_ingestion", current_timestamp()) \
               .withColumn("fichier_source", lit(file_name))
        
        # Sauvegarde
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB_BRONZE}.{table_name}")
        print(f"✅ Succès : {DB_BRONZE}.{table_name}")
        
    except Exception as e:
        print(f"❌ Erreur sur {file_name} : {str(e)}")

# ==========================================
# LANCEMENT GLOBAL
# ==========================================
print("🚀 Démarrage de l'ingestion finale...\n")

ingest_excel_to_bronze("stations_maroc.xlsx", "raw_stations")
ingest_excel_to_bronze("rgph2024_communes.xlsx", "raw_communes")
ingest_excel_to_bronze("tmja_fort_trafic.xlsx", "raw_trafic_tmja")
ingest_excel_to_bronze("resultat_trafic_final.xlsx", "raw_trafic_final")
ingest_excel_to_bronze("hcp_indicateurs_web.xlsx", "raw_hcp_web")
ingest_excel_to_bronze("hcp_pauvrete_brut.xlsx", "raw_hcp_pauvrete")
ingest_excel_to_bronze("parc_automobile_2023.xlsx", "raw_parc_auto")

print("\n🎉 Terminé ! Vérifie tes tables Bronze.")

In [0]:
%pip install pdfplumber

In [0]:
dbutils.library.restartPython()

In [0]:
%run "../00_Configuration/00_parametres"

In [0]:
from pyspark.sql.functions import col, explode, current_timestamp, lit
import pdfplumber
import pandas as pd
import re

print("📦 Ingestion des derniers fichiers complexes...\n")

# --- A. GEOJSON (Routes du Maroc) ---
try:
    # Lecture du GeoJSON en mode multiligne
    df_json = spark.read.option("multiline", "true").json(f"{PATH_BRUT}routes_principales_maroc.geojson")
    
    # Extraction des propriétés et de la géométrie
    df_routes = df_json.select(explode(col("features")).alias("f")) \
        .select(
            col("f.properties.osm_id").alias("id_route"),
            col("f.properties.fclass").alias("type_route"),
            col("f.properties.ref").alias("nom_route"),
            col("f.geometry.coordinates").cast("string").alias("coordonnees_brutes")
        )
    
    df_routes.write.format("delta").mode("overwrite").saveAsTable(f"{DB_BRONZE}.raw_geo_routes")
    print("✅ Succès : raw_geo_routes (GeoJSON) créé.")
except Exception as e:
    print(f"❌ Erreur GeoJSON : {e}")

# --- B. PDF (Recueil Trafic 2023) ---
try:
    pdf_path = f"{PATH_BRUT}recueil_trafic_2023.pdf".replace("dbfs:", "")
    
    with pdfplumber.open(pdf_path) as pdf:
        # On cible la page 32 (index 31) qui contient souvent les synthèses par section
        page = pdf.pages[31]
        table = page.extract_table()
        
        if table:
            # Conversion en DataFrame Pandas puis Spark
            pdf_pd = pd.DataFrame(table[1:], columns=[f"col_{i}" for i in range(len(table[0]))])
            df_pdf = spark.createDataFrame(pdf_pd.astype(str))
            
            df_pdf.withColumn("date_ingestion", current_timestamp()) \
                  .withColumn("fichier_source", lit("recueil_trafic_2023.pdf")) \
                  .write.format("delta").mode("overwrite").saveAsTable(f"{DB_BRONZE}.raw_trafic_pdf_extraits")
            print("✅ Succès : raw_trafic_pdf_extraits créé.")
except Exception as e:
    print(f"❌ Erreur PDF : {e}")

In [0]:
%sql
-- Liste toutes les tables du schéma Bronze
SHOW TABLES IN afriquia_lakehouse.bronze;

In [0]:
# Cellule pour le fichier HTML
try:
    path_html = f"{PATH_BRUT}carte_routes_maroc.html".replace("dbfs:", "")
    with open(path_html, 'r', encoding='utf-8') as f:
        html_content = f.read()
    
    # On crée une petite table avec le contenu complet
    df_html = spark.createDataFrame([(html_content, "carte_routes_maroc.html")], ["html_raw", "source"])
    df_html.write.format("delta").mode("overwrite").saveAsTable(f"{DB_BRONZE}.raw_html_routes")
    print("✅ Succès : raw_html_routes créé.")
except Exception as e:
    print(f"❌ Erreur HTML : {e}")

In [0]:
import pdfplumber

def ingest_all_pdfs(pdf_list):
    for pdf_name in pdf_list:
        try:
            path_local = f"{PATH_BRUT}{pdf_name}".replace("dbfs:", "")
            text_data = []
            with pdfplumber.open(path_local) as pdf:
                # On extrait le texte des 10 premières pages (plus léger pour la Bronze)
                for i in range(min(10, len(pdf.pages))):
                    page_text = pdf.pages[i].extract_text()
                    if page_text:
                        text_data.append((pdf_name, i+1, page_text))
            
            if text_data:
                table_name = "raw_pdf_" + pdf_name.replace(" ", "_").replace(".pdf", "").lower()[:20]
                df_pdf = spark.createDataFrame(text_data, ["fichier", "page", "contenu"])
                df_pdf.write.format("delta").mode("overwrite").saveAsTable(f"{DB_BRONZE}.{table_name}")
                print(f"✅ PDF Ingestéré : {table_name}")
        except Exception as e:
            print(f"❌ Erreur sur {pdf_name} : {e}")

# Liste de tes PDF à traiter
mes_pdfs = [
    "Annuaire Statistique du Maroc, année 2022.pdf",
    "cartographie_pauvrete_2025.pdf",
    "Prospective_Maroc_2030.pdf",
    "rapport_adm_2022.pdf",
    "ennvm_2022_2023.pdf"
]

ingest_all_pdfs(mes_pdfs)